# ChargeGrid Intelligence - Sprint 04
Execute as células na ordem. O projeto será clonado diretamente do GitHub. As respostas e as notas são geradas por modelos reais do Gemini. O relatório numérico exige que os dois conjuntos de testes terminem.

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess
import tempfile
import os
pasta = Path(tempfile.mkdtemp(prefix='chargegrid_sprint04_')) / 'projeto'
subprocess.run(['git', 'clone', '--quiet', 'https://github.com/woowoo88/SPRINT3_IA.git', str(pasta)], check=True)
os.chdir(pasta)
print('Projeto pronto para instalar.')

In [ ]:
# Preserva requisitos de autenticação/rede do ambiente Colab instalado.
from importlib.metadata import requires, PackageNotFoundError
from packaging.requirements import Requirement
from pathlib import Path
restricoes = []
for pacote in ('google-colab', 'google-adk'):
    try:
        for raw in requires(pacote) or []:
            req = Requirement(raw)
            if req.name.lower().replace('_', '-') in {'google-auth', 'requests', 'websockets'} and (req.marker is None or req.marker.evaluate()):
                restricoes.append(f'{req.name}{req.specifier}')
    except PackageNotFoundError:
        pass
Path('/tmp/chargegrid_constraints.txt').write_text('\n'.join(restricoes))
%pip install -q -r requirements-sprint04.txt -c /tmp/chargegrid_constraints.txt

In [ ]:
import os
from getpass import getpass
from google.colab import userdata
try:
    chave = userdata.get('GOOGLE_API_KEY')
except Exception:
    chave = getpass('Chave Gemini (entrada oculta): ')
os.environ['GOOGLE_API_KEY'] = chave
os.environ.pop('GEMINI_API_KEY', None)
del chave
modelo = input('ID do modelo Gemini do agente disponível na sua conta: ').strip()
juiz = input('ID do modelo Gemini avaliador disponível na sua conta: ').strip()
if not modelo or not juiz:
    raise ValueError('Informe os dois IDs de modelo.')

In [ ]:
from sprint04.avaliar import executar
from pathlib import Path
from datetime import datetime, timezone
destino = Path('resultados') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
resultado = executar(modelo, juiz, destino, repeticoes=1)
print('Situação:', resultado['status'])
print('Classificação:', resultado['classificacao'] or 'Inconclusiva por casos com erro')
import shutil
shutil.make_archive('/content/resultados_sprint04', 'zip', destino)
files.download('/content/resultados_sprint04.zip')